# Athens Air Observatory — Final Project
**Guilhem LE DU · Anais KAMTCHOUANG NONO · Yutong CAO · Soukaina BAHAMI**

Repository: https://github.com/guylemme/Data_Science_Athens

Dataset: [Air Quality Monitoring in European Cities — Athens](https://www.kaggle.com/datasets/yekenot/air-quality-monitoring-in-european-cities).
1,726,464 hourly station observations, 19 variables, 64 stations, May 2020–May 2023.
Time: Date; space: latitude, longitude, station; analytical variables: PM2.5, PM10, NO₂, O₃ and weather.
The original notebooks are preserved. This notebook runs the integrated analysis and builds the offline interactive dashboard.
Instructor dataset validation is not evidenced here. No external enrichment dataset is used.


In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display, IFrame
from analysis_pipeline import main as run_analysis
from build_dashboard import build_dashboard
ROOT = Path.cwd()
CSV = ROOT / "data/athens_data.csv"


## 1. Data exploration and preparation
Load the full CSV. Inspect dimensions, types, ranges and missingness. Dates are parsed; tiny negative precipitation is clipped at zero. Negative or infinite analyzed pollutant concentrations become missing. High positive concentrations are retained, and duplicate station timestamps are rejected to avoid inflated hourly counts. Functions are documented in `analysis_pipeline.py`.


In [2]:
def explore_source(csv_path):
    """Input: original CSV path. Output: shape and per-column quality/range table."""
    df = pd.read_csv(csv_path)
    summary = pd.DataFrame({"dtype": df.dtypes.astype(str), "missing": df.isna().sum()})
    summary["missing_percent"] = 100 * summary["missing"] / len(df)
    for column in df.columns:
        values = df[column].dropna()
        summary.loc[column, "min"] = str(values.min()) if len(values) else None
        summary.loc[column, "max"] = str(values.max()) if len(values) else None
    return df.shape, summary

shape, exploration = explore_source(CSV)
print("Dataset dimensions:", shape)
display(exploration)


Dataset dimensions: (1726464, 19)


,dtype,missing,missing_percent,min,max
Date,object,0,0.000000,2020-05-01 00:00:00,2023-05-29 23:00:00
Latitude,float64,0,0.000000,37.813873,38.15
Longitude,float64,0,0.000000,23.538,23.95
station_name,object,0,0.000000,ARISTOTELOUS,THRAKOMAKEDONES
Wind-Speed (U),float64,8573,0.496564,-5.614837717914926,7.170836954246558
Wind-Speed (V),float64,8573,0.496564,-9.550297667781436,10.079717749660205
Dewpoint Temp,float64,8573,0.496564,-10.90312412375551,23.212686721097263
Soil Temp,float64,8573,0.496564,1.957827166689972,37.04616906650417
Total Percipitation,float64,8573,0.496564,-3.4694469519536165e-18,0.0394762287216171
Vegitation (High),float64,8573,0.496564,0.4204858974588402,2.011186618981905


## 2. Four information queries

**Query 1 — station comparison:** group hourly observations by station. Calculate mean PM2.5 and NO₂, maximum PM2.5 and the count of hourly PM2.5 observations above the exploratory reference of 25 µg/m³. The reference is not a regulatory compliance test.

**Query 2 — frequent patterns:** calculate hourly wind magnitude √(U²+V²), aggregate variables per station-day, and discretize at Q1 and Q3. High PM2.5 is a relative category (>Q3), not a health guideline. Remove days missing analysis variables. Apriori support ≥5%, maximum itemset length 4, confidence ≥10%; keep high PM2.5 as the exact consequent. Support=P(A∩B), confidence=P(B|A), lift=P(B|A)/P(B). Associations do not imply causation.

**Query 3 — forecasting:** average daily station means equally across available stations. Hold out the last 30 days. Fit additive damped Holt-Winters with weekly seasonality using the training period only; compare with a weekly seasonal-naive reference using MAE=mean|y−ŷ| and RMSE=√mean(y−ŷ)². Refit on all history for the following 30 days. Predictions are clipped at zero. No prediction intervals; performance is assessed on one holdout.

**Query 4 — spatial clustering:** DBSCAN groups station coordinates using haversine distance (radians), radius 2 km and minimum neighborhood size 3. Label −1 indicates geographical noise, not erroneous data. Compare mean station PM2.5 across clusters; concentration does not determine membership.


In [3]:
def main(csv_path=CSV):
    """Input: Athens CSV. Output: four-query results and offline dashboard path."""
    results = run_analysis(csv_path, ROOT / "results/analysis.json")
    dashboard = build_dashboard(ROOT / "results/analysis.json", ROOT / "Soukaina_Bahami_dashboard.html")
    return results, dashboard

if __name__ == "__main__":
    results, dashboard_path = main()


Four analyses written to C:\Users\yubao\Documents\CODEX\validation\athens-review-20261123\results\analysis.json (1,726,464 original observations).


In [4]:
display(pd.DataFrame(results["station_stats"]).head(15))
display(pd.DataFrame(results["rules"]))
display(pd.DataFrame(results["temporal"]["metrics"]))
display(pd.DataFrame(results["spatial"]["summary"]))


,station_name,pm25_mean,pm25_max,no2_mean,pm25_exceedance_hours,latitude,longitude
0,ATH-ENVICARE-0,23.033377,1748.715385,32.078386,8333,37.976730,23.729800
1,ATHINAS,23.033377,1748.715385,29.610266,8333,37.978000,23.727000
2,ATH-ENVICARE-3,22.429105,544.514286,NaN,7703,37.989967,23.708792
3,PERISTERI,20.922076,483.409091,25.153783,6822,38.021000,23.688000
4,ATH-ENVICARE-2,20.724213,510.044828,19.223274,6567,38.049720,23.757725
5,PANACEA_029,20.312720,370.696552,42.621106,6828,38.005460,23.732340
6,PANACEA_019,19.845114,364.520690,25.124082,6157,38.034657,23.702152
7,PANACEA_039,19.497467,889.393333,28.225540,7134,37.970810,23.748861
8,PANACEA_020,19.472653,209.755556,19.223274,6508,38.090866,23.725220
9,CleanAir in Greece - Thiseio,19.137648,378.367857,NaN,5995,37.974033,23.718441


,antecedents,consequent,support,confidence,lift
0,[Wind_Speed_Level_Low],PM25_Level_High,0.112202,0.448790,1.804260
1,"[Humidity_Level_Medium, Wind_Speed_Level_Low]",PM25_Level_High,0.054425,0.424308,1.705836
2,[Humidity_Level_High],PM25_Level_High,0.086429,0.345720,1.389888
3,[Temperature_Level_Low],PM25_Level_High,0.077782,0.311497,1.252301
4,[Temperature_Level_High],PM25_Level_High,0.063156,0.252471,1.015003
5,[Wind_Speed_Level_Medium],PM25_Level_High,0.116659,0.233320,0.938012
6,[Humidity_Level_Medium],PM25_Level_High,0.115904,0.231812,0.931946
7,[Temperature_Level_Medium],PM25_Level_High,0.107802,0.215541,0.866532
8,"[Humidity_Level_Medium, Wind_Speed_Level_Medium]",PM25_Level_High,0.051002,0.204687,0.822897
9,"[Humidity_Level_Medium, Temperature_Level_Medium]",PM25_Level_High,0.056632,0.177612,0.714050


,Model,MAE,RMSE
0,Holt_Winters,4.039497,4.834095
1,Seasonal_naive,4.339170,5.412688


,cluster,number_of_stations,mean_PM25,min_station_PM25,max_station_PM25
0,4,4,19.049957,17.685822,20.922076
1,2,18,17.981903,13.042619,23.033377
2,3,3,16.216233,13.260216,20.724213
3,1,3,15.918589,13.853156,18.754278
4,0,4,14.309036,11.228887,15.718113


## 3. Interactive dashboard and interpretation
The HTML embeds all results and Plotly itself, so it opens without Internet or a server. Use its four analysis tabs to visualize each query; filter station metrics, association rules, forecast views and geographical clusters. “Data & methods” records missingness and limitations. The figures can be exported as PNG.
The dashboard is also a standalone submission artifact; the original 459 MB CSV stays out of Git.


In [5]:
display(IFrame(src=dashboard_path.name, width="100%", height=900))


## 4. Presentation and reproducibility
Use `Soukaina_Bahami_slides.pdf` and `presentation/speaker_notes.md` for the five-minute presentation. Run `python -m unittest discover -s tests -v` to check analytical invariants. Download source data using the README instructions. Source SHA-256 and computation time are stored in results metadata. Minor optimizer/version differences can change forecast errors slightly from earlier notebook outputs.
